In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from lib.latex_plots import configure_matplotlib

# configure_matplotlib()
EXTERNAL = Path('../data/external')

df = pd.read_parquet(EXTERNAL / "papers_cleaned.gencat.parquet")
df['date'] = pd.to_datetime(df['date'])
df['year'] = df['date'].dt.year.astype(int) # from the date, we only want the year
df = df[df['year']>= 1990] # remove rows with NaN in the 'date' column
df = df[df['year'] < 2026] # remove rows with NaN in the 'date' column

# FILTER DUPLICATED DOIS-----------------------------------------------
df_unics = df.drop_duplicates(subset=["doi"])
#----------------------------------------------------------------------

# Count publications per year
publicacions_per_any = df_unics['year'].value_counts().sort_index()

years = publicacions_per_any.index
count = publicacions_per_any.values

In [6]:
import numpy as np
from tqdm import tqdm
# configure_matplotlib()
# --- Supongamos que ya tienes years y count ---
# years = [...]
# count = [...]
SLIDES = Path('../plots/slides')
OUT_DIR = SLIDES / "year_frames"
OUT_DIR.mkdir(parents=True, exist_ok=True)

n_frames = len(years)
last5 = set(range(n_frames - 5, n_frames))   # índices de los últimos 5 años
major_years = [y for y in years if y % 5 == 0]
minor_years = [y for y in years if y % 5 != 0]

for frame in tqdm(range(n_frames), desc="Generando frames"):
    fig, ax = plt.subplots(figsize=(10, 7))
    fig.patch.set_facecolor('white')
    ax.set_facecolor('#fafafa')

    for side in ('top', 'right'):
        ax.spines[side].set_visible(False)
    ax.grid(axis='y', linestyle='--', alpha=0.4)
    ax.set_axisbelow(True)

    # Barras: solo las visibles hasta el frame actual
    heights = [count[i] if i <= frame else 0 for i in range(n_frames)]
    bars = ax.bar(years, heights,
                  color='#3b82f6', edgecolor='#1e3a8a', width=0.7)

    # Etiquetas numéricas SOLO en los últimos 5 años
    # for i in last5:
    #     if i <= frame:
    #         ax.text(years[i], count[i], str(count[i]),
    #                 ha='center', va='bottom',
    #                 fontsize=9, color='#1e3a8a', fontweight='bold')

    # Ejes fijos (sin título)
    ax.set_xlabel('Year', fontsize=20)
    ax.set_ylabel(r'# Papers', fontsize=20)
    # Ticks mayores (con etiqueta) cada 5 años
    ax.set_xticks(major_years)
    ax.set_xticklabels(major_years, fontsize=20, rotation=45, ha='right')
    ax.set_yticks([5000, 10000, 15000, 20000])
    ax.set_yticklabels(['5k', '10k', '15k', '20k'], fontsize=20)
    # Ticks menores (sin etiqueta) para los años restantes
    ax.set_xticks(minor_years, minor=True)

    # Estética de los ticks
    ax.tick_params(axis='x', which='major', length=6, width=1)
    ax.tick_params(axis='x', which='minor', length=3, width=0.8, color='#888')
    ax.set_ylim(0, max(count) * 1.02)
    ax.set_xlim(min(years) - 0.6, max(years) + 0.6)

    fig.tight_layout()
    fig.savefig(f'{OUT_DIR}/frame_{frame:03d}.png', dpi=210)
    plt.close(fig)

print(f'✅ {n_frames} frames guardados en ./{OUT_DIR}/')

Generando frames: 100%|██████████| 36/36 [00:03<00:00, 10.16it/s]

✅ 36 frames guardados en ./../plots/slides/year_frames/


In [3]:
from PIL import Image
import glob
import os

FRAMES_DIR = SLIDES/'year_frames'
OUTPUT_GIF = SLIDES/'papers_evolution.gif'
FPS = 5          # frames por segundo → duración de cada frame = 1000/FPS ms
LOOP = 0         # 0 = bucle infinito

# --- Cargar todos los frames en orden ---
paths = sorted(glob.glob(os.path.join(FRAMES_DIR, 'frame_*.png')))
if not paths:
    raise FileNotFoundError(f'No se encontraron PNGs en {FRAMES_DIR}/')

frames = [Image.open(p).convert('RGB') for p in paths]

# --- Guardar como GIF ---
duration_ms = int(1000 / FPS)
frames[0].save(
    OUTPUT_GIF,
    save_all=True,
    append_images=frames[1:],
    duration=duration_ms,
    loop=False,
    optimize=False,        # reduce tamaño
    disposal=2,           # limpia el frame anterior (evita "fantasmas")
)

# --- Liberar memoria ---
for f in frames:
    f.close()

size_kb = os.path.getsize(OUTPUT_GIF) / 1024
print(f'✅ {OUTPUT_GIF} generado con {len(frames)} frames ({size_kb:.1f} KB)')

✅ ../plots/slides/papers_evolution.gif generado con 36 frames (2151.1 KB)
